# Random Regions

Authors: [Serge Rey](https://github.com/sjsrey), [James Gaboardi](https://github.com/jGaboardi)

## Introduction

In this notebook we demonstrate how to use `spopt` with empirical and synthetic examples by:

1. Evaluating the properties of a predefined regionalization scheme.
2. Exploring the `max_swaps` parameter on a simple lattice. 

`spopt` offers functionality to generate random regions based on user-defined constraints. There are three optional parameters to constrain the regionalization: number of regions, cardinality, and contiguity. The default case simply takes a list of area IDs and randomly selects the number of regions and then allocates areas to each region. The user can also pass a vector of integers to the cardinality parameter to designate the number of areas to randomly assign to each region. The contiguity parameter takes a spatial weights object and uses that to ensure that each region is made up of spatially contiguous areas. When the contiguity constraint is enforced, it is possible to arrive at infeasible solutions; the `maxiter` parameter can be set to make multiple attempts to find a feasible solution. 

We begin with importing the relevant packages:

In [ ]:
%config InlineBackend.figure_format = "retina"
%load_ext watermark
%watermark

In [ ]:
import warnings

import geopandas
import inequality
import libpysal
import matplotlib
import matplotlib.pyplot as plt
import numpy
import pandas
import seaborn

import spopt
from spopt.region import RandomRegion, RandomRegions

RANDOM_SEED = 12345

%watermark -w
%watermark -iv

## 1. Evaluating Regional Partitions: The case of Mexican state incomes

We will be using the built-in example data set `mexico` from the [libpysal](https://pysal.org/libpysal/) package. First, we get a high-level overview of the example:

In [ ]:
libpysal.examples.explain("mexico")

In [ ]:
mdf = geopandas.read_file(libpysal.examples.get_path("mexicojoin.shp"))

In [ ]:
mdf.head()

This data set records per capital Gross Domestic Product (PCGDP) for the decades 1940-2000 for 32 Mexican states.

In [ ]:
mdf.columns

We can plot the spatial distribution of regional incomes in the last decade of the sample:

In [ ]:
mdf.plot(
    figsize=(8, 6),
    column="PCGDP2000",
    scheme="quantiles",
    cmap="Blues",
    edgecolor="grey",
    legend=True,
    legend_kwds={"fmt": "{:.0f}"},
).axis("off")

Here we see the north-south divide in the Mexican space economy. This pattern has been the subject of much recent research, and a common methodological strategy in these studies is to *partition* the states into mutually exclusive and exhaustive regions. The motivation is similar to that in traditional clustering where one attempts to group like objects together. Here the regional sets should be composed of states that are more similar to other members of the same set, and distinct from those states in other sets.

Our dataset has a number of regionalization schemes from the published literature. Here we will focus on one `HANSON03`. We can plot this partition using a categorical scheme:

In [ ]:
mdf.plot(figsize=(8, 6), column="HANSON03", categorical=True).axis("off");

Here the different colors symbolize membership in a different region. We can examine the cardinality structure of this partition:

In [ ]:
cards = mdf.groupby(by="HANSON03").count().NAME.values.tolist()
cards

Thus, we see there six regions (sets) of different sizes, the smallest region has only two states, while the largest (in a set-theory sense) has 10 states. We can explore the regional definitions in tabular form:

In [ ]:
mdf[["NAME", "HANSON03", "PCGDP2000"]].sort_values(by="HANSON03")

## How good is `HANSON03` as a partition?

One question we might ask about the partition, is how good of a job does it do at capturing the spatial distribution of incomes in Mexico? To answer this question, we are going to leverage the random regions functionality in `spopt`.

### A random partition respecting cardinality constraints

We will first create a new random partition that has the same cardinality structure/distribution as the `HANSON03` partition:

In [ ]:
ids = mdf.index.values.tolist()

In [ ]:
numpy.random.seed(RANDOM_SEED)
rrmx = RandomRegion(ids, num_regions=6, cardinality=cards)

This creates a new `RandomRegion` instance. One of its attributes is the definition of the regions in the partition:

In [ ]:
rrmx.regions

This will have the same cardinality as the `HANSON03` partition:

In [ ]:
{len(region) for region in rrmx.regions} == set(cards)

A simple helper function let's us attach the region labels for this new partition into the dataframe:

In [ ]:
def region_labels(df, solution, name="labels_"):
    n, k = df.shape
    labels_ = numpy.zeros((n,), int)
    for i, region in enumerate(solution.regions):
        labels_[region] = i
    df[name] = labels_

And, then we can visualize the new partition:

In [ ]:
region_labels(mdf, rrmx, name="rrmx")
mdf.plot(figsize=(8, 6), column="rrmx", categorical=True).axis("off");

While the `rrmx` partition and the `HANSON03` partitions have identical cardinality structures, their spatial distributions are radically different. The `rrmx` map looks nothing like the `HANSON03` map.

One of the key differences is that the sets in the `HANSON03` partition are each spatially connected components while this is not the case for the "regions" in the `rrmx` partition.

### A random partition respecting cardinality and contiguity constraints

A second way to form a random partition is to add an additional constraint in the form of the spatial connectivity between the states. To do so, we will construct a spatial weights object using the `Queen` contiguity criterion:

In [ ]:
with warnings.catch_warnings(record=True) as w:
    w = libpysal.weights.Queen.from_dataframe(mdf)

and then add this as an additional parameter to instantiate a new `RandomRegion`:

In [ ]:
numpy.random.seed(RANDOM_SEED)
rrmxc = RandomRegion(ids, num_regions=6, cardinality=cards, contiguity=w)

In [ ]:
rrmxc.regions

This will also have the same cardinality as the `HANSON03` partition:

In [ ]:
{len(region) for region in rrmxc.regions} == set(cards)

But, more importantly, we see the partition yields spatially connected components for the regions:

In [ ]:
region_labels(mdf, rrmxc, name="rrmxc")
mdf.plot(figsize=(8, 6), column="rrmxc", categorical=True).axis("off");

## Comparing Partitions

We now can get back to the question of how good the `HANSON03` partition does in capturing the spatial structure of state incomes in Mexico. Two alternative random partitions have been constructed using `RandomRegion` and we can compare these with the `HANSON03` partition:

In [ ]:
f, axs = plt.subplots(1, 3, figsize=(12, 8))
ax1, ax2, ax3 = axs

mdf.plot(column="HANSON03", categorical=True, ax=ax1)
ax1.set_axis_off()
ax1.set_aspect("equal")
ax1.set_title("HANSON03")

mdf.plot(column="rrmx", categorical=True, ax=ax2)
ax2.set_axis_off()
ax2.set_aspect("equal")
ax2.set_title("Cardinality Constrained")

mdf.plot(column="rrmxc", categorical=True, ax=ax3)
ax3.set_axis_off()
ax3.set_aspect("equal")
ax3.set_title("Cardinality and Contiguity Constrained")

plt.subplots_adjust(wspace=-0.2);

In order to judge the `HANSON03` regions against those from the two random solutions, we need some objective function to serve as our benchmark. Given the interest in the spatial distribution of incomes, we might ask if a partition has minimized the internal heterogeneity of the incomes for states belonging to each set?

We can use the PySAL [inequality](https://inequality.readthedocs.io/en/latest/) package to implement this measure:

Let's focus on the last year in the sample and pull out the state incomes:

In [ ]:
y = mdf.PCGDP2000

The [TheilD](https://inequality.readthedocs.io/en/latest/generated/inequality.theil.TheilD.html#inequality.theil.TheilD) statistic decomposes the inequality of the 32 state incomes into two parts:

- inequality between states belonging to different regions (inequality between regions)
- inequality between states belonging to the same region (inequality within regions)

We can calculate this for the original partition:

In [ ]:
t_hanson = inequality.theil.TheilD(y, mdf.HANSON03)

The overall level of inequality is:

In [ ]:
t_hanson.T

which is decomposed into the "between-regions" component:

In [ ]:
t_hanson.bg

and the "within-regions" component:

In [ ]:
t_hanson.wg

For this partition, the two components are roughly equal. How does this compare to a random partition? Well, for a random partition with the same cardinality structure as `HANSON03` we can carry out the same decomposition:

In [ ]:
t_rrmx = inequality.theil.TheilD(y, mdf.rrmx)

The level of overall inequality will be the same:

In [ ]:
t_rrmx.T

However, the decomposition is very different now, with the within component being much larger than the between component.

In [ ]:
t_rrmx.bg

In [ ]:
t_rrmx.wg

How about when both cardinality and contiguity are taken into account when developing the random partition? We can pass in the partition definition for this solution into the decomposition:

In [ ]:
t_rrmxc = inequality.theil.TheilD(y, mdf.rrmxc)

Again, the total inequality remains the same:

In [ ]:
t_rrmxc.T

and the within-region component is even larger than either the `HANSON03` and `rrmx` partitions:

In [ ]:
t_rrmxc.bg

In [ ]:
t_rrmxc.wg

### Generating a Reference Distribution

Both of the comparison partitions are examples of random partitions, subject to matching the cardinality (`rrmx`) or cardinality and contiguity constraints (`rmxc`). As such, the generated partitions should be viewed as samples from the population of all possible random partitions that also respect those constraints. Rather than relying on a single draw from each of these distributions, as we have done up to now, if we sample from these distributions, we can develop an empirical reference distribution to compare the original `HANSON03` partition against. In other words, we can ask if the latter partition is performing any better than a random partition that respects the cardinality and contiguity constraints.

We can do this by adding in a `permutations` argument to the class `RandomRegions`. Note the plural here in that this class is intended to generate `permutations` solutions for the random regions that respect the relevant cardinality and contiguity constraints. We will generate `permutations=99` solutions and calculate the inequality decomposition for each of the 99 partitions, extracting the `wg` component to develop the reference distribution
against which we can evaluate the observed `wg` value from the `HANSON03` partition:

In [ ]:
numpy.random.seed(RANDOM_SEED)
rrmxcs = RandomRegions(
    ids, num_regions=6, cardinality=cards, contiguity=w, permutations=99
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    wg = []
    for i, solution in enumerate(rrmxcs.solutions_feas):
        name = f"rrmxc_{i}"
        region_labels(mdf, solution, name=name)
        wg.append(inequality.theil.TheilD(y, mdf[name]).wg)
wg = numpy.array(wg)

The mean of the within-region inequality component is:

In [ ]:
wg.mean()

which is larger than the observed value for `HANSON03`:

In [ ]:
t_hanson.wg

Examining the empirical distribution of the within-region inequality component for the random partitions:

In [ ]:
rdf = pandas.DataFrame(data=wg, columns=["Within-region inequality"])

In [ ]:
_ = seaborn.displot(
    rdf, kind="kde", legend=False, x="Within-region inequality", rug=True
);

provides a more comprehensive benchmark for evaluating the `HANSON03` partition. We see that the observed within-region component is extreme relative to what we would expect if the partitions were formed randomly subject to the
cardinality and contiguity constraints:

In [ ]:
t_hanson.wg

In [ ]:
wg.min()

We could even formalize this statement by developing a pseudo p-value for the observed component:

In [ ]:
pvalue = (1 + (wg <= t_hanson.wg[0]).sum()) / 100
pvalue

which would lead us to conclude that the observed value is significantly different from what would be expected under a random region null. 

So, yes,the `HANSON03` partition does appear to be capturing the spatial pattern of incomes in Mexico in the sense that its larger within-region inequality component is not due to random chance.

---------------------------------

## 2. Demonstrating the `max_swaps` parameter

Generate a 20 x 20 lattice in `spenc` with synthetic labels & plot.

In [ ]:
hori, vert = 20, 20
n_polys = hori * vert
gdf = spopt.region.spenclib.utils.lattice(hori, vert)
numpy.random.seed(RANDOM_SEED)
gdf["synth_data"] = numpy.random.laplace(100, 10, n_polys)
gdf.head()

In [ ]:
plt.rcParams["figure.figsize"] = [10, 10]
gdf.plot(
    column="synth_data",
    scheme="quantiles",
    cmap="Blues",
    edgecolor="grey",
    legend=True,
    legend_kwds={"fmt": "{:.2f}"},
);

Set the number of desired regions to 4.

In [ ]:
n_regions = 4

Define IDs & set cardinality to 100 areas per region.

In [ ]:
ids = gdf.index.values.tolist()
cards = [int(n_polys / n_regions)] * n_regions
cards

Generate a `RandomRegion` instance (constraining only region count and cardinality) and plot.

In [ ]:
numpy.random.seed(RANDOM_SEED)
rrlat = RandomRegion(ids, num_regions=n_regions, cardinality=cards)

In [ ]:
region_labels(gdf, rrlat, name="rrlat")
gdf.plot(column="rrlat", cmap="tab20");

Create a `Queen` weights object to define spatial contiguity.

In [ ]:
with warnings.catch_warnings(record=True) as w:
    w = libpysal.weights.Queen.from_dataframe(gdf)

Iterate over `max_swaps=[100, 350, 650, 999]` for 4 `RandomRegion` instances.

In [ ]:
for swaps in [100, 350, 650, 999]:
    swap_str = f"rrlatms{swaps}"
    numpy.random.seed(RANDOM_SEED)
    result = RandomRegion(
        ids,
        num_regions=n_regions,
        cardinality=cards,
        contiguity=w,
        compact=True,
        max_swaps=swaps,
    )
    region_labels(gdf, result, name=swap_str)

Assign labels and titles, then plot.

In [ ]:
labels = [
    ["synth_data", "rrlatms100", "rrlatms650"],
    ["rrlat", "rrlatms350", "rrlatms999"],
]
titles = [
    ["Data", "Max 100 Swaps", "Max 650 Swaps"],
    ["Unconstrained", "Max 350 Swaps", "Max 999 Swaps"],
]

In [ ]:
rows, cols = 2, 3
f, axs = plt.subplots(rows, cols, figsize=(9, 7.5))
for i in range(rows):
    for j in range(cols):
        label, title = labels[i][j], titles[i][j]
        _ax = axs[i, j]
        if i == j and i == 0:
            kws = {"scheme": "quantiles", "cmap": "Blues", "ec": "grey"}
            gdf.plot(column=label, ax=_ax, **kws)
        else:
            gdf.plot(column=label, ax=_ax, cmap="tab20")
        _ax.set_title(title)
        _ax.set_axis_off()
        _ax.set_aspect("equal")
plt.subplots_adjust(wspace=1, hspace=0.5)
plt.tight_layout()

As shown here, the results obtained from setting the maximum number of swaps clearly vary.